# Initial data gathering and preparation 

First notebook to run 

In [ ]:
import os
import pandas as pd
from google.cloud import bigquery

import matplotlib.pyplot as plt
import seaborn as sns


from Helper_Functions import *
from helping_variables import *

## Get the initial data from the all-of-us laboratory measurements database

In [ ]:

# 1. Workspace setup
YOUR_WORKSPACE_PROJECT = os.environ.get("GOOGLE_PROJECT")
DATASET = DATASET_ID # "<Project ID>.R2025Q4R6"

client = bigquery.Client(project=YOUR_WORKSPACE_PROJECT)

# 2. Pipeline Inputs
num_tests = 300000
# target_protein = "hba1c"  # Options: "hba1c", "insulin", "gdf15", "glucose", "leptin", "adiponectin"
target_protein = "insulin"  # Options: "hba1c", "insulin", "gdf15", "glucose", "leptin", "adiponectin"

# 3. RE2-Compliant Target Rules
TARGET_RULES = {
    "hba1c": {
        "include": r"(hba1c|hemoglobin a1c|glycated hemoglobin)",
        "exclude": r"(glucose|eag|calculated|mean|average)",
    },
    "insulin": {
        "include": r"\binsulin\b",
        "exclude": r"(igf|binding|growth)",
    },
    "gdf15": {
        "include": r"(gdf-15|gdf15|growth differentiation factor 15)",
        "exclude": None,
    },
    "adiponectin": {"include": r"adiponectin", "exclude": None},
    "leptin": {"include": r"leptin", "exclude": None},
    "glucose": {
        "include": r"glucose",
        "exclude": r"(urine|urur|tolerance|challenge|dipstick)",
    },
}

rule = TARGET_RULES.get(
    target_protein.lower(),
    {"include": rf"\b{target_protein}\b", "exclude": None},
)

include_pattern = rule["include"]
exclude_clause = (
    f"AND NOT REGEXP_CONTAINS(LOWER(concept_name), r'{rule['exclude']}')"
    if rule["exclude"]
    else ""
)

# 4. SQL Query with Name Harmonization & Mathematical Unit Normalization
query = rf"""
WITH target_concepts AS (
    SELECT concept_id, concept_name
    FROM `{DATASET}.concept`
    WHERE domain_id = 'Measurement'
      AND REGEXP_CONTAINS(LOWER(concept_name), r'{include_pattern}')
      {exclude_clause}
),

anchor_encounters AS (
    SELECT DISTINCT 
        m.person_id, 
        m.measurement_date,
        m.visit_occurrence_id
    FROM 
        `{DATASET}.measurement` m
    JOIN 
        target_concepts tc ON m.measurement_concept_id = tc.concept_id
    WHERE 
        m.value_as_number IS NOT NULL
    ORDER BY 
        m.person_id, m.measurement_date
    LIMIT {num_tests}
),

raw_labs AS (
    SELECT 
        m.person_id,
        m.measurement_date,
        m.measurement_datetime,
        m.visit_occurrence_id,
        m.visit_detail_id,
        m.provider_id,
        c.concept_name,
        m.value_as_number,
        m.unit_source_value,
        m.unit_concept_id,
        uc.concept_name AS unit_standard_name
    FROM 
        `{DATASET}.measurement` m
    JOIN 
        anchor_encounters ae 
        ON m.person_id = ae.person_id 
        AND m.measurement_date = ae.measurement_date
    JOIN 
        `{DATASET}.concept` c ON m.measurement_concept_id = c.concept_id
    LEFT JOIN 
        `{DATASET}.concept` uc ON m.unit_concept_id = uc.concept_id
    WHERE 
        c.domain_id = 'Measurement'
        AND m.value_as_number IS NOT NULL
),

standardized_labs AS (
    SELECT 
        *,
        -- Step 1: Assign Standard Test Name
        CASE 
            WHEN REGEXP_CONTAINS(LOWER(concept_name), r'(hba1c|hemoglobin a1c|glycated hemoglobin)') AND NOT REGEXP_CONTAINS(LOWER(concept_name), r'(glucose|eag|calculated|mean|average)') THEN 'HbA1c'
            WHEN REGEXP_CONTAINS(LOWER(concept_name), r'(gdf-15|gdf15|growth differentiation factor 15)') THEN 'GDF15'
            WHEN LOWER(concept_name) LIKE '%adiponectin%' THEN 'Adiponectin'
            WHEN LOWER(concept_name) LIKE '%leptin%' THEN 'Leptin'
            WHEN LOWER(concept_name) LIKE '%insulin%' AND NOT REGEXP_CONTAINS(LOWER(concept_name), r'(igf|binding|growth)') THEN 'Insulin'
            WHEN LOWER(concept_name) LIKE '%glucose%' AND NOT REGEXP_CONTAINS(LOWER(concept_name), r'(urine|tolerance|challenge|dipstick)') THEN 'Glucose'
            ELSE INITCAP(LOWER(concept_name))
        END AS laboratory_test_name,
        
        -- Clean string for unit matching
        LOWER(COALESCE(unit_standard_name, unit_source_value, '')) AS clean_unit
    FROM raw_labs
)

SELECT 
    person_id,
    measurement_date,
    measurement_datetime,
    visit_occurrence_id,
    visit_detail_id,
    provider_id,
    laboratory_test_name,
    value_as_number AS raw_value,
    COALESCE(unit_standard_name, unit_source_value) AS raw_unit,
    
    -- Step 2: Assign Standardized Target Unit per Test
    CASE laboratory_test_name
        WHEN 'HbA1c' THEN '%'
        WHEN 'Insulin' THEN 'uIU/mL'
        WHEN 'Glucose' THEN 'mg/dL'
        WHEN 'GDF15' THEN 'pg/mL'
        WHEN 'Leptin' THEN 'ng/mL'
        WHEN 'Adiponectin' THEN 'ug/mL'
        ELSE clean_unit
    END AS target_unit,
    
    -- Step 3: Perform Unit Math Conversions
    CASE laboratory_test_name
        -- HbA1c Math: mmol/mol (IFCC) to % (NGSP)
        WHEN 'HbA1c' THEN 
            CASE 
                WHEN REGEXP_CONTAINS(clean_unit, r'(mmol/mol|mmol/m)') THEN ROUND((value_as_number / 10.929) + 2.15, 2)
                ELSE value_as_number
            END
            
        -- Insulin Math: pmol/L or ng/mL to uIU/mL
        WHEN 'Insulin' THEN 
            CASE 
                WHEN REGEXP_CONTAINS(clean_unit, r'(pmol/l|pmol)') THEN ROUND(value_as_number / 6.945, 2)
                WHEN REGEXP_CONTAINS(clean_unit, r'(ng/ml)') THEN ROUND(value_as_number * 24.1, 2)
                ELSE value_as_number
            END

        -- Glucose Math: mmol/L to mg/dL
        WHEN 'Glucose' THEN 
            CASE 
                WHEN REGEXP_CONTAINS(clean_unit, r'(mmol/l)') THEN ROUND(value_as_number * 18.0182, 1)
                ELSE value_as_number
            END

        -- GDF15 / Leptin / Adiponectin prefix scaling (pg/mL <-> ng/mL <-> ug/mL)
        WHEN 'GDF15' THEN 
            CASE 
                WHEN REGEXP_CONTAINS(clean_unit, r'(ng/ml)') THEN ROUND(value_as_number * 1000, 2)
                ELSE value_as_number
            END
            
        ELSE value_as_number
    END AS value_normalized

FROM standardized_labs
"""

df_long = client.query(query).to_dataframe()
df_long["visit_occurrence_id"] = df_long["visit_occurrence_id"].fillna(-1)
print(f"Retrieved {len(df_long):,} normalized lab results.")
df_long.sample(20)

### test for which units the laboratory test has, and whether the distributions are similar?

In [ ]:
# prot = "HbA1c"
prot = "Insulin"
# prot = "Glucose"

df_long.loc[df_long["laboratory_test_name"]==prot]
# df_long.loc[df_long["laboratory_test_name"]==prot]["visit_occurrence_id"].value_counts()
# df_long.loc[df_long["laboratory_test_name"]==prot]["visit_detail_id"].value_counts()
# df_long.loc[df_long["laboratory_test_name"]==prot]["provider_id"].value_counts()
# df_long.loc[df_long["laboratory_test_name"]==prot]["unit_source_value"].value_counts()
# df_long.loc[df_long["laboratory_test_name"]==prot]["unit_concept_id"].value_counts()
df_long.loc[df_long["laboratory_test_name"]==prot]["raw_unit"].value_counts()
# df_long.loc[df_long["laboratory_test_name"]==prot]["target_unit"].value_counts()

In [ ]:
df_long.loc[(df_long["laboratory_test_name"]==prot) & (df_long["raw_unit"]=="percent")]["value_normalized"].describe()

df_long.loc[(df_long["laboratory_test_name"]==prot) & (df_long["raw_unit"]=="micro-international unit per milliliter")]["value_normalized"].describe()


In [ ]:
df_long.loc[df_long["laboratory_test_name"]==prot]["value_normalized"].describe()

In [ ]:
df_long.loc[df_long["laboratory_test_name"]==prot]["raw_value"].describe()

In [ ]:
# df_long.loc[(df_long["laboratory_test_name"]==prot) & (df_long["raw_unit"]=="milligram per deciliter")].sample(5)

#### plot all units distributions

In [ ]:

prot_data = df_long.loc[df_long["laboratory_test_name"]==prot].copy()
prot_data = prot_data[(prot_data["raw_value"]>df_long["raw_value"].quantile(0.10)) & (prot_data["raw_value"]<df_long["raw_value"].quantile(0.90))]
prot_data

# 1. Filter data for missing and non-positive values
clean_data = prot_data[prot_data["raw_value"].notna()].copy()
log_data = clean_data[clean_data["raw_value"] > 0].copy()

# 2. Set up a large 2x1 figure canvas (width=18, height=6)
fig, axes = plt.subplots(2, 1, figsize=(20, 15))

# 3. Subplot 1: Normal / Linear Scale
sns.histplot(
    data=clean_data,
    x="raw_value",
    hue="raw_unit",
    # binrange=(prot_data["raw_value"].quantile(0.10),prot_data["raw_value"].quantile(0.90)),
    log_scale=False,
    kde=True,
    ax=axes[0],
)
axes[0].set_title("Linear Scale", fontsize=14, fontweight="bold")

# 4. Subplot 2: Log Scale
sns.histplot(
    data=log_data,
    x="raw_value",
    hue="raw_unit",
    log_scale=True,
    kde=True,
    ax=axes[1],
)
axes[1].set_title("Log Scale", fontsize=14, fontweight="bold")

# 5. Move legends outside the plot boundaries to fit large unit lists
sns.move_legend(axes[0], "upper left", bbox_to_anchor=(1.02, 1))
sns.move_legend(axes[1], "upper left", bbox_to_anchor=(1.02, 1))

plt.tight_layout()
plt.show()

## First filter, drop all laboratory tests with too few occurrences

In [ ]:
# 2.5 Filter out tests with fewer than N occurrences
N = 10
test_counts = df_long['laboratory_test_name'].value_counts()
valid_tests = test_counts[test_counts >= N].index

print(test_counts)
print(valid_tests)


In [ ]:
test_counts >= num_tests
print("mesurments with more tests then num_tests")
test_counts.loc[test_counts >= num_tests]

In [ ]:
df_long = df_long[df_long['laboratory_test_name'].isin(valid_tests)].reset_index(drop=True)
print(f"Retrieved {len(df_long):,} lab results after keeping tests with >= {N} occurrences.")

## For any laboratory test, keep only the main test unit and the units with a similar distribution to it

In [ ]:

for test in df_long['laboratory_test_name'].unique():
    print(f" test = {test}")
    print(f" number of tests = {len(df_long.loc[df_long["laboratory_test_name"]==test])}")
    raw_units = df_long.loc[df_long["laboratory_test_name"]==test]["raw_unit"].value_counts().index
    print(len(raw_units))
    print(raw_units)

    break

In [ ]:
df_long.loc[df_long["laboratory_test_name"]==test]["raw_unit"].unique()

In [ ]:
# -------------------------------------------------------------------------
# Run Audit
# -------------------------------------------------------------------------
df_audit = audit_lab_unit_distributions_dual_scale(
    df_long, min_sample_size=15, emd_q_threshold=0.25, emd_s_threshold=2, jsd_threshold=0.25, use_rule="emd_and_jsd",
    trim_outliers=True, quantile_trim_bounds=(0.05, 0.95),
)


# Filter for persistent true anomalies
df_true_alerts = df_audit[df_audit["final_alert"] == True]

print(f"Audited {len(df_audit)} unit pairs.")
print(f"Flagged {len(df_true_alerts)} true distribution mismatches.\n")

df_audit[
    [
        "laboratory_test_name",
        "comparison_unit",
        "alert_linear",
        "alert_log",
        "final_alert",
    ]
]

### audit results for our main 3 laboratory tests

In [ ]:
# prot = "HbA1c"
# prot = "Insulin"
prot = "Glucose"

df_audit.loc[df_audit["laboratory_test_name"].isin(["HbA1c", "Insulin", "Glucose"])]

### remove laboratory tests with alerts

In [ ]:
# View the exact test-unit combinations removed
df_audit.loc[df_audit["final_alert"]==True]
df_audit.loc[
    df_audit["final_alert"] == True,
    ["laboratory_test_name", "reference_unit", "comparison_unit", "comp_count"],
]

In [ ]:
F_flagged_units = df_audit.loc[
    df_audit["final_alert"] == False, ["laboratory_test_name", "comparison_unit"]
].rename(columns={"comparison_unit": "raw_unit"})
F_flagged_index = pd.MultiIndex.from_frame(F_flagged_units)
long_index = pd.MultiIndex.from_frame(
    df_long[["laboratory_test_name", "raw_unit"]]
)
# print(f"length of False flagged units = {len(F_flagged_units):,}")
print(f"length of False flagged units = {len(df_long[long_index.isin(F_flagged_index)].copy()):,}")

T_flagged_units = df_audit.loc[
    df_audit["final_alert"] == True, ["laboratory_test_name", "comparison_unit"]
].rename(columns={"comparison_unit": "raw_unit"})
T_flagged_index = pd.MultiIndex.from_frame(T_flagged_units)
long_index = pd.MultiIndex.from_frame(
    df_long[["laboratory_test_name", "raw_unit"]]
)
# print(f"length of True flagged units = {len(T_flagged_units):,}")
print(f"length of True flagged units = {len(df_long[long_index.isin(T_flagged_index)].copy()):,}")

print(
    f"Original rows: {len(df_long):,} "
)

In [ ]:
# 1. Extract problematic (test_name, raw_unit) pairs from audit alerts
flagged_units = df_audit.loc[
    df_audit["final_alert"] == True, ["laboratory_test_name", "comparison_unit"]
].rename(columns={"comparison_unit": "raw_unit"})

# 2. Create MultiIndices for set comparison
flagged_index = pd.MultiIndex.from_frame(flagged_units)
long_index = pd.MultiIndex.from_frame(
    df_long[["laboratory_test_name", "raw_unit"]]
)

# 3. Drop flagged rows from df_long
df_long_cleaned = df_long[~long_index.isin(flagged_index)].copy()

# Summary
dropped_rows = len(df_long) - len(df_long_cleaned)
print(
    f"Original rows: {len(df_long):,} | Remaining rows: {len(df_long_cleaned):,}"
)
print(
    f"Dropped {dropped_rows:,} rows across {len(flagged_units)} anomalous unit variants."
)

## Again, drop all laboratory tests with too few occurrences 

if our previous filter created new ones

In [ ]:
# 2.5 Filter out tests with fewer than N occurrences
N = 10
test_counts = df_long_cleaned['laboratory_test_name'].value_counts()
valid_tests = test_counts[test_counts >= N].index

print(test_counts)
print(valid_tests)


In [ ]:
test_counts >= num_tests
print("mesurments with more tests then num_tests")
test_counts.loc[test_counts >= num_tests]

In [ ]:
df_long_cleaned = df_long_cleaned[df_long_cleaned['laboratory_test_name'].isin(valid_tests)].reset_index(drop=True)
print(f"Retrieved {len(df_long_cleaned):,} lab results after keeping tests with >= {N} occurrences.")

## See statistics on multiple occurrences of the same laboratory test in the same testing keys


In [ ]:
# Define your encounter index columns (adjust if using measurement_date instead)
# index_cols = ['person_id', 'visit_occurrence_id', 'measurement_datetime']
index_cols = ['person_id', 'measurement_date', 'visit_occurrence_id']
test_col = 'laboratory_test_name'

# 1. Count readings per test per encounter
encounter_counts = (
    df_long_cleaned.groupby(index_cols + [test_col])
    .size()
    .reset_index(name='reading_count')
)

# 2. Filter to encounters with > 1 value for the same test
multi_value_occurrences = encounter_counts[encounter_counts['reading_count'] > 1]

# 3. Aggregate by test to rank tests by multi-reading frequency
multi_value_summary = (
    multi_value_occurrences.groupby(test_col)
    .agg(
        encounters_with_duplicates=('reading_count', 'count'),
        max_readings_in_single_encounter=('reading_count', 'max')
    )
    .sort_values(by='encounters_with_duplicates', ascending=False)
    .reset_index()
)

print(f"Found {len(multi_value_summary)} tests with multiple values per index:\n")
# print(multi_value_summary.head(20))
multi_value_summary.head(20)

## Pivot the data from long format to wide format

In [ ]:
# 3. Pivot into wide format
df_wide = df_long_cleaned.pivot_table(
    index=['person_id', 'measurement_date', 'visit_occurrence_id'], 
    columns='laboratory_test_name', 
    values='value_normalized',
    aggfunc='mean'
).reset_index()

df_wide.columns.name = None

# 4. Reorder columns to put target proteins up front
base_cols = ['person_id', 'measurement_date', 'visit_occurrence_id']
target_proteins = ['Glucose', 'HbA1c', 'Insulin', 'Adiponectin', 'Leptin', 'IGFBP1', 'IGFBP2', 'Resistin', 'Glucagon', 'GDF15']

present_targets = [col for col in target_proteins if col in df_wide.columns]
other_cols = [col for col in df_wide.columns if col not in base_cols and col not in target_proteins]

df_wide = df_wide[base_cols + present_targets + other_cols]

print(f"\nFinal Table: {df_wide.shape[0]:,} encounters x {df_wide.shape[1]} total columns.")
print(f"Target proteins positioned at start: {present_targets}")
df_wide.head()

In [ ]:
# 1. Count non-null observations per lab in df_wide
total_counts = df_wide.notna().sum()

# 2. Map total counts onto multi_value_summary using test names
multi_value_summary['total_encounters'] = multi_value_summary['laboratory_test_name'].map(total_counts)

# 3. Calculate percentage / proportion of duplicates
multi_value_summary['pct_duplicate'] = (
    multi_value_summary['encounters_with_duplicates'] / multi_value_summary['total_encounters']
)

multi_value_summary.head(20)

### how many tests do we have from our interesting list

In [ ]:
df_wide.loc[:,present_targets]
df_wide.loc[:,present_targets].copy().notna()
df_wide.loc[:,present_targets].copy().notna().sum()


### wide data info

In [ ]:
df_wide.info()

In [ ]:

print(f"number of final rows: {len(df_wide)}")
df_wide.describe()


In [ ]:

if target_protein == "hba1c":
    print(f"target_protein = {target_protein}")
    num_samples = df_wide.loc[:,"HbA1c"].copy().notna().sum()
    print(f"num_samples = {num_samples}")
    print(f"num missing = {df_wide.loc[:,"HbA1c"].copy().isna().sum()}")
if target_protein == "insulin":
    print(f"target_protein = {target_protein}")
    num_samples = df_wide.loc[:,"Insulin"].copy().notna().sum()
    print(f"num_samples = {num_samples}")
    print(f"num missing = {df_wide.loc[:,"Insulin"].copy().isna().sum()}")


## Save the data for later work

In [ ]:


# 1. Convert BigQuery's 'dbdate' type to standard pandas datetime
print("Converting date columns to standard pandas datetime...")
if 'measurement_date' in df_long_cleaned.columns:
    df_long_cleaned['measurement_date'] = pd.to_datetime(df_long_cleaned['measurement_date'])

if 'measurement_date' in df_wide.columns:
    df_wide['measurement_date'] = pd.to_datetime(df_wide['measurement_date'])

# Define file names
file_long = f"data\\aou_{target_protein}_labs_long_cleaned.parquet"
file_wide = f"data\\aou_{target_protein}_labs_wide_cleaned.parquet"

# 2. Save DataFrames to Parquet format
print("Saving DataFrames to Parquet format...")
df_long_cleaned.to_parquet(file_long, index=False, engine='pyarrow')
df_wide.to_parquet(file_wide, index=False, engine='pyarrow')

# 3. Print file sizes to confirm
size_long_mb = os.path.getsize(file_long) / (1024 * 1024)
size_wide_mb = os.path.getsize(file_wide) / (1024 * 1024)

print("\n Data successfully saved to your workspace persistent disk!")
print(f"📁 {file_long}: {size_long_mb:.2f} MB")
print(f"📁 {file_wide}: {size_wide_mb:.2f} MB")

## Chack the saved data files

In [ ]:

# Load saved Parquet files
file_long = f"data\\aou_{target_protein}_labs_long_cleaned.parquet"
file_wide = f"data\\aou_{target_protein}_labs_wide_cleaned.parquet"

print("Loading saved Parquet files...")
df_long = pd.read_parquet(file_long, engine='pyarrow')
df_wide = pd.read_parquet(file_wide, engine='pyarrow')

print(f"\n Loaded df_long: {df_long.shape[0]:,} rows x {df_long.shape[1]} columns")
print(f" Loaded df_wide: {df_wide.shape[0]:,} rows x {df_wide.shape[1]} columns")

df_wide.head()

## What is the main test unit for each laboratory test

In [ ]:
laboratory_test_name = "Insulin"
test_units = df_long.loc[df_long["laboratory_test_name"]==laboratory_test_name]["raw_unit"].value_counts()
test_units.index[0:6]

In [ ]:
df_long

# main_tests_units = pd.Dataframe()
main_tests_units = pd.Series()

df_wide.columns[3:]
for laboratory_test_name in df_wide.columns[3:]:
    test_units = df_long.loc[df_long["laboratory_test_name"]==laboratory_test_name]["raw_unit"].value_counts()
    
    if len(test_units) == 0:
        print(f"{laboratory_test_name} has no units at all!!!!")
        main_tests_units.loc[laboratory_test_name] = None
    elif test_units.index[0] != 'No matching concept':
        main_tests_units.loc[laboratory_test_name] = test_units.index[0]
    elif len(test_units) > 1:
        main_tests_units.loc[laboratory_test_name] = test_units.index[1]
    elif len(test_units) == 1:
        print(f"{laboratory_test_name} has no given units!!")
        main_tests_units.loc[laboratory_test_name] = test_units.index[0]    
        
main_tests_units

In [ ]:
test_units

In [ ]:
laboratory_test_name

In [ ]:
df_long.loc[df_long["laboratory_test_name"]==laboratory_test_name]